# Chapter 11 — The Milky Way disk in Gaia data: the solar neighbourhood and the rotation curve

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/11_solar_neighbourhood.ipynb)

**Goal.** Apply the tools of Chapters 4–10 to real stars. We take Gaia DR3 stars with full six-dimensional
phase-space coordinates, put them in the Galactocentric frame, and look at the structures that the models of
the earlier chapters predict — and some that they do not: moving groups, the phase spiral, the rotation curve
and the distribution of stars in action space.

**You will learn**
1. to convert Gaia observables (position, parallax, proper motion, line-of-sight velocity) to Galactocentric
   positions and velocities with AGAMA, and which assumptions about the Sun enter;
2. to see the **moving groups** in velocity space and the **phase spiral** (Chapter 6) in the $(z, v_z)$ plane,
   and to test that a feature is not noise;
3. to measure the **rotation curve** from red giants with the Jeans equation (Chapter 7), reproducing a published
   result, and to fit a dark halo to it — and see which halo properties the data do and do not constrain;
4. to compute **actions** for hundreds of thousands of stars (Chapter 9) and see the moving groups as ridges.

**Data.** Five files prepared for this tutorial are on Zenodo, https://doi.org/10.5281/zenodo.23627795
(licence CC BY-NC 4.0); its README describes the selections and every column. This chapter uses three of them:

| file | content | size |
|---|---|---|
| `solar_neighbourhood` | 4.6 M Gaia DR3 stars with line-of-sight velocities in a cylinder around the Sun (0.5 kpc radius, $\pm$1.2 kpc tall) | 274 MB |
| `disc_giants` | 44 784 red giants with spectrophotometric parallaxes (Hogg, Eilers & Rix 2019), APOGEE DR17 velocities, Gaia DR3 proper motions | 6 MB |
| `rotation_curves` | the published $v_c(R)$ of Eilers et al. (2019) and Ou et al. (2024) | 5 kB |

The first download of `solar_neighbourhood` takes a few minutes; afterwards the files are read from the `data/` folder.

**Units.** kpc, km/s, $M_\odot$; actions in kpc km/s; angles on the sky in degrees, proper motions in mas/yr.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object

import os, hashlib, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd                                # tables; pd.read_parquet needs pyarrow (preinstalled on Colab)
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib.patheffects as pe               # white outlines that keep markers visible on any background
from scipy.ndimage import gaussian_filter
from scipy.optimize import least_squares

# Zenodo record 10.5281/zenodo.23627795 (version 1.0). Its files never change, so we can check them by MD5 checksum.
ZENODO = 'https://zenodo.org/records/23627795/files/{}?download=1'
MD5 = {'solar_neighbourhood': 'f6119c98ae50ded14fab91d1d4c071fd', 'halo_gse': '1c0533ce89d6d95ff67406dcf2a95f31',
       'disc_giants': '12f6a9ca0ac3d8ecdcd71988267b8ea7', 'rotation_curves': 'a41c95e4ecc31b3ab3bde99880736c6e',
       'dsph_gaia': '0b135e7f08816209b379f892196307b2'}


def load_data(name, cache=Path('data')):
    """Download data/<name>.parquet from Zenodo once, check its MD5, and return it as a pandas DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} from Zenodo ...')
        urllib.request.urlretrieve(ZENODO.format(path.name), path)
    md5 = hashlib.md5(path.read_bytes()).hexdigest()
    assert md5 == MD5[name], f'{path} is corrupt (md5 {md5}); delete it and rerun'
    return pd.read_parquet(path)


def vcirc(pot, R):
    """Midplane circular speed sqrt(-R F_R) (Chapter 0 §0.4)."""
    R = np.atleast_1d(R)
    return np.sqrt(-R * pot.force(np.column_stack([R, 0 * R, 0 * R]))[:, 0])

## 11.1 From Gaia observables to Galactocentric coordinates

Gaia measures, for each star, the sky position $(\alpha, \delta)$, the parallax $\varpi$, the proper motion
$(\mu_{\alpha*}, \mu_\delta)$ (where $\mu_{\alpha*} = \mu_\alpha\cos\delta$), and, for the brighter stars, the
line-of-sight velocity $v_{\rm los}$. To do dynamics we need positions and velocities relative to the Galactic centre.
Four steps:

1. **distance** $d = 1/\varpi$ (kpc for $\varpi$ in mas). This is a good estimate only when the parallax is precise;
   our file keeps stars with $\varpi/\sigma_\varpi > 10$, so the error is below 10%;
2. **tangential velocity** $v_t = 4.74047\, \mu\, d$ km/s for $\mu$ in mas/yr and $d$ in kpc
   (1 mas/yr at 1 kpc is 1 AU/yr = 4.74047 km/s);
3. **rotate** from the equatorial (ICRS) frame to Galactic coordinates $(l, b)$;
4. **shift** to the Galactic centre: subtract the Sun's position and add the Sun's velocity.

Step 4 needs the **solar parameters**. They are measured, not known exactly, and every result below depends on them:

| parameter | value | source |
|---|---|---|
| distance to the Galactic centre $R_0$ | 8.122 kpc | GRAVITY Collaboration (2018), orbit of the star S2 around Sgr A* |
| height above the midplane $z_\odot$ | 20.8 pc | Bennett & Bovy (2019) |
| Galactocentric velocity $(v_x, v_y, v_z)_\odot$ | (12.9, 245.6, 7.78) km/s | Drimmel & Poggio (2018); the default of `astropy` |

AGAMA does steps 3–4 with `agama.transformCelestialCoords` and `agama.getGalactocentricFromGalactic`.
**Its convention** (the same as astropy's `Galactocentric` frame): the Sun is at $x = -R_0$, $y = 0$, $z = z_\odot$;
$x$ points from the Sun to the Galactic centre and $y$ in the direction of Galactic rotation. Because the Sun sits at
negative $x$ and moves towards $+y$, its angular momentum $L_z = x v_y - y v_x$ is **negative**: the disk rotates
clockwise seen from the North Galactic Pole. We therefore define the rotation velocity with a minus sign,
$v_\phi = -L_z/R$, so that it is positive (about +230 km/s) for disk stars.

In [ ]:
R0, z_sun = 8.122, 0.0208                  # kpc
v_sun = (12.9, 245.6, 7.78)                 # km/s, Galactocentric (towards the centre, rotation, North pole)
K = 4.740470463533348                       # km/s per (mas/yr * kpc)


def galactocentric(ra, dec, parallax, pmra, pmdec, vlos):
    """Gaia observables (deg, deg, mas, mas/yr, mas/yr, km/s) -> Galactocentric x, y, z [kpc] and vx, vy, vz [km/s]."""
    # ICRS -> Galactic: AGAMA wants angles in radians; it rotates the proper motions along with the positions
    l, b, pml, pmb = agama.transformCelestialCoords(agama.fromICRStoGalactic, np.radians(ra), np.radians(dec), pmra, pmdec)
    d = 1 / parallax                        # kpc
    # proper motions enter in km/s/kpc (= K mas/yr), so that pm * distance is a velocity in km/s
    return agama.getGalactocentricFromGalactic(l, b, d, pml * K, pmb * K, vlos,
                                               galcen_distance=R0, z_sun=z_sun, galcen_v_sun=v_sun)


def cylindrical(x, y, z, vx, vy, vz):
    """Cylindrical radius and velocities; v_phi > 0 in the direction of Galactic rotation (see text)."""
    R = np.hypot(x, y)
    vR = (x * vx + y * vy) / R              # projection on the unit vector (x, y)/R
    vphi = -(x * vy - y * vx) / R           # -L_z / R
    return R, vR, vphi

In [ ]:
sn = load_data('solar_neighbourhood')
print(f'{len(sn):,} stars; columns: {list(sn.columns)}')
sn.describe().T[['mean', 'min', 'max']]       # .T: one row per column

Each row is a star: `parallax` and its error, proper motions, `radial_velocity` (the line-of-sight velocity; the
file's README explains a small correction applied to it), magnitude `phot_g_mean_mag`, colour `bp_rp`, and the
metallicity `feh` and surface gravity `logg` estimated from Gaia's low-resolution spectra (Andrae et al. 2023; NaN where
unavailable). We convert all 4.6 million stars at once — AGAMA's routines take whole numpy arrays.

In [ ]:
cols = ['ra', 'dec', 'parallax', 'pmra', 'pmdec', 'radial_velocity']
x, y, z, vx, vy, vz = galactocentric(*[sn[c].to_numpy(dtype=float) for c in cols])   # float32 in the file -> float64
R, vR, vphi = cylindrical(x, y, z, vx, vy, vz)
d = 1 / sn.parallax.to_numpy(dtype=float)
print(f'median v_R = {np.median(vR):.2f}, v_phi = {np.median(vphi):.2f}, v_z = {np.median(vz):.2f} km/s')

**Check against an independent implementation.** astropy's `Galactocentric` frame does the same transformation with
its own code. We compare on 20 000 random stars, with the same solar parameters.

In [ ]:
import astropy.coordinates as coord, astropy.units as u

i = np.random.default_rng(1).choice(len(sn), 20000, replace=False)
frame = coord.Galactocentric(galcen_distance=R0 * u.kpc, z_sun=z_sun * u.kpc,
                             galcen_v_sun=coord.CartesianDifferential(v_sun * u.km / u.s))
c = coord.SkyCoord(ra=sn.ra.values[i] * u.deg, dec=sn.dec.values[i] * u.deg, distance=d[i] * u.kpc,
                   pm_ra_cosdec=sn.pmra.values[i] * u.mas / u.yr, pm_dec=sn.pmdec.values[i] * u.mas / u.yr,
                   radial_velocity=sn.radial_velocity.values[i] * u.km / u.s).transform_to(frame)
dpos = np.abs(c.cartesian.xyz.to_value(u.kpc) - np.array([x[i], y[i], z[i]])).max()
dvel = np.abs(c.velocity.d_xyz.to_value(u.km / u.s) - np.array([vx[i], vy[i], vz[i]])).max()
print(f'AGAMA vs astropy: max position difference {dpos * 1e3:.1e} pc, max velocity difference {dvel:.1e} km/s')
assert dpos < 1e-5 and dvel < 0.01

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist2d(x, y, bins=200, norm=LogNorm(), cmap='Greys')
ax[0].plot(-np.sqrt(R0**2 - z_sun**2), 0, 'r*', ms=12, label='Sun')
ax[0].set_xlabel('$x$ [kpc]'); ax[0].set_ylabel('$y$ [kpc]'); ax[0].set_aspect('equal'); ax[0].legend()
ax[1].hist2d(y, z, bins=200, norm=LogNorm(), cmap='Greys')
ax[1].set_xlabel('$y$ [kpc]'); ax[1].set_ylabel('$z$ [kpc]')
plt.tight_layout(); plt.show()

assert abs(np.median(vR)) < 5 and abs(np.median(vz)) < 1 and 215 < np.median(vphi) < 235

**Figure 11.1.** The 4.6 million stars of the sample, as log-scaled counts. Left: projected on the Galactic plane, $(x, y)$, with the Sun marked. Right: in the vertical plane $(y, z)$.

The two codes agree to a few thousandths of a parsec and a thousandth of a km/s, so the transformation is right; what
remains uncertain are its inputs (the solar parameters, the parallaxes). The sample fills a cylinder around the Sun
(Figure 11.1) that is denser near the plane, where most stars are. The median velocities are what we expect for disk stars:
$v_\phi \approx 226$ km/s, $v_R$ and $v_z$ near zero. The median $v_R$ of about $-2$ km/s is partly a choice: change the
solar $v_x$ by 2 km/s and it vanishes. The median $v_z$ is within 1 km/s of zero by construction of $v_{z,\odot}$.

## 11.2 Velocity space: moving groups

A smooth equilibrium disk has a smooth velocity distribution near the Sun (Chapter 10 built one). The real
distribution, for the 620 000 stars within 200 pc, is not smooth. It is broken into streams called **moving groups**,
named after star clusters found in them long before Gaia. To bring out the structure, the right panel of Figure 11.2 subtracts a
smoothed version of the (log) histogram — a high-pass filter.

We mark four classical groups at the heliocentric velocities $(U, V)$ of Dehnen (1998), as tabulated by
Antoja et al. (2008, Table 3). $U$ points to the Galactic centre and $V$ in the direction of rotation; in our
variables $v_R = -(U + v_{x,\odot})$ and $v_\phi = V + v_{y,\odot}$ (outward is $-U$).

In [ ]:
near = d < 0.2
groups = {'Hyades': (-40, -20), 'Pleiades': (-12, -22), 'Sirius': (9, 3), 'Hercules': (-32, -50)}   # (U, V) km/s
groups = {k: (V + v_sun[1], -(U + v_sun[0])) for k, (U, V) in groups.items()}                      # -> (v_phi, v_R)

rng_v = [[150, 300], [-100, 100]]                # v_phi and v_R ranges [km/s]
label_at = {'Hyades': (250, 75), 'Pleiades': (180, -60), 'Sirius': (275, -75), 'Hercules': (170, 70)}  # label positions
H, ve_phi, ve_R = np.histogram2d(vphi[near], vR[near], bins=150, range=rng_v)   # bins of 1 x 1.33 km/s; ve_*: edges
lnH = np.log(H + 1)
excess = lnH - gaussian_filter(lnH, 8)           # log(counts) minus a version smoothed over 8 km/s

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
ax[0].hist2d(vphi[near], vR[near], bins=300, range=rng_v, norm=LogNorm(), cmap='Greys')
ax[0].set_title(f'stars within 200 pc: {near.sum():,}')
ax[1].imshow(excess.T, origin='lower', extent=[*rng_v[0], *rng_v[1]], aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax[1].set_title('log counts minus smoothed (red = excess)')
for a in ax:
    for name, (vp, vr) in groups.items():
        # open circle at the group's velocity; label in a white box outside the dense region, joined by a line
        a.plot(vp, vr, 'o', ms=14, mfc='none', mec='k', mew=2, path_effects=[pe.withStroke(linewidth=4, foreground='w')])
        a.annotate(name, (vp, vr), label_at[name], fontsize=11, fontweight='bold', ha='center', va='center',
                   bbox=dict(boxstyle='round', fc='w', ec='k'),
                   arrowprops=dict(arrowstyle='-', color='k', lw=1.5, shrinkB=8,
                                   path_effects=[pe.withStroke(linewidth=4, foreground='w')]))
    a.set_xlabel('$v_\\phi$ [km/s]'); a.set_ylabel('$v_R$ [km/s]')
plt.tight_layout(); plt.show()

# the filtered map at each group's position (average of the 5x5 bins around it): positive = overdensity
for name, (vp, vr) in groups.items():
    ip, ir = np.searchsorted(ve_phi, vp) - 1, np.searchsorted(ve_R, vr) - 1   # pixel containing the position
    groups[name] = excess[ip - 2:ip + 3, ir - 2:ir + 3].mean()
    print(f'{name:9s}: excess in log counts {groups[name]:+.2f}')
assert all(e > 0.1 for e in groups.values())

**Figure 11.2.** Velocities $(v_\phi, v_R)$ of the 620 000 stars within 200 pc. Left: log-scaled counts. Right: log counts minus a smoothed version (red = excess). Circles mark four classical moving groups at the velocities of Dehnen (1998).

All four classical groups sit on local overdensities of the Gaia data (Figure 11.2, right), although they were found in Hipparcos data
of only ~10 000 stars. With Gaia they turn out to be parts of long, curved **arches**, and there are many more. Moving
groups are not dissolving clusters: their stars have a wide range of ages and metallicities (Antoja et al. 2008).
Most are thought to be orbits trapped near **resonances** with the Galactic bar or spiral arms — for example Hercules,
whose stars lag the Sun's rotation by about 50 km/s. An axisymmetric equilibrium model like those of Chapter 10 cannot
produce them; §11.5 shows what they look like in action space.

## 11.3 The phase spiral

Chapter 6 (§6.4) produced a phase spiral by kicking a model disk and letting it phase-mix. Antoja et al. (2018) found
the real one in Gaia DR2. It is clearest not in the star counts but in the **mean rotation velocity** across the
$(z, v_z)$ plane: stars on different turns of the spiral have slightly different $v_\phi$.

The mean $v_\phi$ changes smoothly with the vertical energy (stars with larger vertical excursions rotate more slowly,
the asymmetric drift of Chapter 7). We remove this trend by subtracting the mean $v_\phi$ in rings of the "elliptical
radius" $r_e = \sqrt{(z/0.5\,{\rm kpc})^2 + (v_z/25\,{\rm km\,s^{-1}})^2}$, a rough proxy for the vertical energy. What
is left is the spiral. We keep stars within 0.5 kpc of the Sun's radius (the whole file) and pixels with at least 200 stars.

**Is it real?** Noise also makes patterns. We test this with a **shuffle**: within each ring, give the $v_\phi$ values
to randomly chosen other stars of the same ring. This destroys any spiral but keeps the noise level and the trend.
A real signal makes the data map vary more than the shuffled one.

In [ ]:
re = np.hypot(z / 0.5, vz / 25.)                       # elliptical radius in the (z, v_z) plane
ring = np.digitize(re, np.linspace(0, 3, 61))          # ring index of each star (0.05 wide)
rng_z = [[-1.1, 1.1], [-60, 60]]                        # z [kpc] and v_z [km/s] ranges
N_pix, _, _ = np.histogram2d(z, vz, bins=88, range=rng_z)   # 0.025 kpc x 1.36 km/s pixels


def residual_map(q):
    """Mean of q minus its ring average, in each (z, v_z) pixel; NaN where fewer than 200 stars."""
    ring_mean = np.bincount(ring, q) / np.bincount(ring)     # sum of q per ring / stars per ring
    S, _, _ = np.histogram2d(z, vz, bins=88, range=rng_z, weights=q - ring_mean[ring])
    m = S / np.maximum(N_pix, 1)
    m[N_pix < 200] = np.nan
    return m


rng = np.random.default_rng(0)
vphi_shuffled = vphi.copy()
for k in np.unique(ring):
    members = np.flatnonzero(ring == k)
    vphi_shuffled[members] = vphi[rng.permutation(members)]   # same values, randomly reassigned within the ring

maps = {'data': residual_map(vphi), 'shuffled within rings': residual_map(vphi_shuffled)}
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, (name, m) in zip(ax, maps.items()):
    im = a.imshow(m.T, origin='lower', extent=[*rng_z[0], *rng_z[1]], aspect='auto', cmap='coolwarm', vmin=-4, vmax=4)
    a.set_title(f'{name}: rms {np.nanstd(m):.2f} km/s')
    a.set_xlabel('$z$ [kpc]'); a.set_ylabel('$v_z$ [km/s]')
plt.colorbar(im, ax=ax, label='$v_\\phi$ minus ring mean [km/s]')
plt.show()

ratio = np.nanstd(maps['data']) / np.nanstd(maps['shuffled within rings'])
print(f'rms of the data map / rms of the shuffled map = {ratio:.2f}')
assert ratio > 1.5

**Figure 11.3.** Mean $v_\phi$ minus its ring mean in each $(z, v_z)$ pixel; blank pixels have fewer than 200 stars. Left: the data. Right: the same after shuffling $v_\phi$ among stars of the same ring.

The spiral (Figure 11.3, left) winds about one and a half times from the centre out to $|z| \approx 0.7$ kpc, with an amplitude of a few
km/s. The shuffled map (right) has no structure, and its rms (the noise) is about half that of the data. Two points of
interpretation:

- The spiral is **one-armed** and winds the same way as the model in Chapter 6: the disk was pushed out of vertical
  equilibrium some hundreds of Myr ago and is still phase-mixing. Proposed causes include a passage of the Sagittarius
  dwarf galaxy and the buckling of the bar; this is an open problem.
- Its winding is a **clock** (§6.4), but reading it needs the vertical frequency as a function of amplitude, i.e. the
  potential; the next section measures the potential.

## 11.4 The rotation curve from red giants

Near the Sun, Gaia parallaxes are excellent; across the disk they are not. The 44 784 luminous red giants of
Hogg, Eilers & Rix (2019) have **spectrophotometric parallaxes**, predicted from their APOGEE spectra and photometry
with a data-driven model, precise to about 7% out to 20 kpc. Eilers et al. (2019) used them to measure the rotation
curve. We repeat their measurement with Gaia DR3 proper motions and APOGEE DR17 velocities.

**Method.** The radial Jeans equation of an axisymmetric disk (Chapter 7, §7.5), in the midplane and neglecting the
small $\langle v_R v_z \rangle$ term, gives the circular speed from the moments of the velocities:

$$ v_c^2(R) = \langle v_\phi^2 \rangle - \langle v_R^2 \rangle \left(1 + \frac{\partial \ln \nu}{\partial \ln R} +
   \frac{\partial \ln \langle v_R^2 \rangle}{\partial \ln R}\right). $$

Following Eilers et al.:
- the tracer density $\nu \propto e^{-R/3\,{\rm kpc}}$ is **assumed** ($\partial \ln\nu/\partial\ln R = -R/3$): the APOGEE
  selection function is too complex to measure it; they show this is the largest systematic error, about 2%;
- $\langle v_R^2\rangle$ is measured in 0.5 kpc bins and fitted with an exponential, which gives its derivative;
- the measurement errors inflate the second moments; we subtract each star's velocity error variance, estimated by
  redrawing its parallax, proper motions and line-of-sight velocity from their errors 50 times (a Monte Carlo);
- the selection: within $30°$ of the Sun–centre line, close to the plane ($|z| < 0.5$ kpc or $|z|/R < \tan 6°$),
  $|v_z| < 100$ km/s (no halo stars), and low $\alpha$-abundance ($[{\rm Mg/Fe}] < 0.12$; Eilers used APOGEE's
  $[\alpha/{\rm Fe}]$) — the thin disk, whose small dispersion keeps the correction small.

In [ ]:
gi = load_data('disc_giants')
gi = gi[np.isfinite(gi.vhelio_avg)]                # 25 stars have no APOGEE velocity
print(f'{len(gi):,} giants; columns: {list(gi.columns)}')

obs = lambda df, plx: (df.ra.to_numpy(float), df.dec.to_numpy(float), plx, df.pmra.to_numpy(float),
                       df.pmdec.to_numpy(float), df.vhelio_avg.to_numpy(float))   # galactocentric() arguments
gx, gy, gz, gvx, gvy, gvz = galactocentric(*obs(gi, gi.spec_parallax.to_numpy(float)))
gR, gvR, gvphi = cylindrical(gx, gy, gz, gvx, gvy, gvz)
phi = np.degrees(np.arctan2(gy, -gx))              # azimuth from the Sun-centre line (Sun at 0)

sel = ((np.abs(phi) < 30) & ((np.abs(gz) < 0.5) | (np.abs(gz) / gR < np.tan(np.radians(6))))
       & (np.abs(gvz) < 100) & (gi.mg_fe.to_numpy() < 0.12))
s = gi[sel]
print(f'selected: {sel.sum():,} stars')

# Monte Carlo error propagation: 50 realisations of each star's observables
rng = np.random.default_rng(42)
draws = []
for k in range(50):
    noisy = lambda col, err: s[col].to_numpy(float) + rng.normal(size=len(s)) * s[err].to_numpy(float)
    xv = galactocentric(s.ra.to_numpy(float), s.dec.to_numpy(float), noisy('spec_parallax', 'spec_parallax_error'),
                        noisy('pmra', 'pmra_error'), noisy('pmdec', 'pmdec_error'), noisy('vhelio_avg', 'verr'))
    draws.append(cylindrical(*xv)[1:])             # (v_R, v_phi) of this realisation
draws = np.array(draws)                            # shape (50, 2, N_stars)
# Error variance per star and component. A plain variance is fragile: a redrawn parallax that comes out near zero gives
# an absurd distance and dominates it. Half the 16-84 percentile range is the standard deviation for a Gaussian, robustly.
q16, q84 = np.percentile(draws, [16, 84], axis=0)
err2_vR, err2_vphi = ((q84 - q16) / 2)**2
print(f'median velocity errors: v_R {np.sqrt(np.median(err2_vR)):.1f}, v_phi {np.sqrt(np.median(err2_vphi)):.1f} km/s')

# second moments in 0.5 kpc bins, minus the error variance (Eilers et al. eq. 4)
Rs, vRs, vphis = gR[sel], gvR[sel], gvphi[sel]
edges = np.arange(5, 25.01, 0.5)
N_all = np.histogram(Rs, edges)[0]                 # stars per bin
keep = N_all >= 3                                  # bins with at least 3 stars
mean_in_bins = lambda q: (np.histogram(Rs, edges, weights=q)[0] / np.maximum(N_all, 1))[keep]   # sum / count
N = N_all[keep]
Rb = mean_in_bins(Rs)
VRR = mean_in_bins(vRs**2 - err2_vR)               # <v_R^2>
Vpp = mean_in_bins(vphis**2 - err2_vphi)           # <v_phi^2>

# exponential fit of sqrt(<v_R^2>) in well-populated bins: ln sqrt(VRR) = a - R/h
a1, a0 = np.polyfit(Rb[N > 20], 0.5 * np.log(VRR[N > 20]), 1)
h_sig = -1 / a1
VRR_fit = np.exp(2 * (a0 + a1 * Rb))
vc_ours = np.sqrt(Vpp - VRR_fit * (1 - Rb / 3.0 - 2 * Rb / h_sig))   # dln nu/dlnR = -R/3; dln VRR/dlnR = -2R/h
print(f'scale length of sqrt(<v_R^2>): {h_sig:.0f} kpc (Eilers et al.: about 21 kpc)')

In [ ]:
rc = load_data('rotation_curves')
eil, ou = rc[rc.source == 'Eilers2019'], rc[rc.source == 'Ou2024']

data_dir = os.path.join(os.path.dirname(agama.__file__), 'data')
mcmillan = agama.Potential(os.path.join(data_dir, 'McMillan17.ini'))   # Chapter 4
Rgrid = np.linspace(4, 28, 100)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].plot(Rb, np.sqrt(VRR), 'C0o', ms=4, label='$\\sqrt{\\langle v_R^2\\rangle}$')
ax[0].plot(Rb, np.sqrt(VRR_fit), 'C0-', label=f'exponential, scale {h_sig:.0f} kpc')
ax[0].set_xlabel('$R$ [kpc]'); ax[0].set_ylabel('[km/s]'); ax[0].legend(); ax[0].set_ylim(0)
ax[1].errorbar(eil.R, eil.vc, [eil.vc_err_minus, eil.vc_err_plus], fmt='s', color='C0', ms=4, label='Eilers et al. 2019')
ax[1].errorbar(ou.R, ou.vc, [ou.vc_err_minus, ou.vc_err_plus], fmt='^', color='C1', ms=4, label='Ou et al. 2024')
ax[1].plot(Rb, vc_ours, 'ko', ms=4, label='this notebook (Jeans)')
ax[1].plot(Rb, np.sqrt(Vpp), 'k.', ms=3, alpha=0.5, label='$\\sqrt{\\langle v_\\phi^2\\rangle}$ (no correction)')
ax[1].plot(Rgrid, vcirc(mcmillan, Rgrid), 'C3--', label='McMillan (2017) model')
ax[1].set_xlabel('$R$ [kpc]'); ax[1].set_ylabel('$v_c$ [km/s]'); ax[1].set_ylim(150, 260); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

ok = (Rb > 6) & (Rb < 17) & (N > 50)
dev = vc_ours[ok] / np.interp(Rb[ok], eil.R, eil.vc) - 1
print(f'{ok.sum()} bins at 6-17 kpc: ours / Eilers - 1 ranges from {dev.min():+.3f} to {dev.max():+.3f}')
assert np.all(np.abs(dev) < 0.02)

**Figure 11.4.** Left: $\sqrt{\langle v_R^2\rangle}$ of the selected giants in 0.5 kpc bins (points) and its exponential fit (line). Right: the circular speed from the Jeans equation (black), the uncorrected $\sqrt{\langle v_\phi^2\rangle}$ (grey), the published curves of Eilers et al. (2019, blue squares) and Ou et al. (2024, orange triangles), and McMillan's (2017) model (red dashed).

Between 6 and 17 kpc our curve agrees with Eilers et al. (Figure 11.4, right) within 1% in every bin, although we used newer data
(Gaia DR3 instead of DR2, APOGEE DR17 instead of DR14), slightly different solar parameters, and $[{\rm Mg/Fe}]$ instead
of $[\alpha/{\rm Fe}]$. Our $\langle v_R^2\rangle$ falls more slowly with radius (Figure 11.4, left; scale length about 36 instead of
21 kpc), a reminder that such fits depend on details of the binning and the selection; the effect on $v_c$ is small.
Beyond about 17 kpc there are few stars per bin and the curves scatter.

Things to notice:
- The correction from $\sqrt{\langle v_\phi^2\rangle}$ (grey dots in Figure 11.4) to $v_c$ is 5–15 km/s, growing outwards, for this
  cold population; for the old, hot thick disk it would be several times larger (Chapter 7).
- The curve **declines** gently, by about 1.7 km/s per kpc (Eilers et al.). Ou et al. (2024), with the same method applied to
  120 000 giants with APOGEE DR17 and Gaia DR3 data, find a curve a few km/s higher inside 15 kpc and a steeper
  decline beyond about 20 kpc. The difference is systematic, not statistical — inside 15 kpc the curves differ by
  more than their error bars.
- McMillan's (2017) model, made before these data, lies 3–10% above both beyond 10 kpc (as seen in Chapter 4).

### Fitting a dark halo

What does the rotation curve tell us about the dark matter? Write the potential as fixed **baryons** — the stellar
and gas disks and the bulge of McMillan's model — plus an **NFW halo** whose two parameters we fit: AGAMA's `mass`
parameter of the NFW profile ($M_{\rm NFW} = 4\pi\rho_s r_s^3$; the total NFW mass diverges) and the scale radius
$r_s$. Circular speeds add in quadrature, $v_c^2 = v_{\rm bar}^2 + v_{\rm halo}^2$. We fit the Eilers et al. points by
least squares (`scipy.optimize.least_squares` minimises the sum of squared residuals in units of the error bars).

In [ ]:
import configparser
cfg = configparser.ConfigParser()
cfg.read(os.path.join(data_dir, 'McMillan17.ini'))         # the .ini file of Chapter 4: one section per component
baryons = agama.Potential(*[dict(cfg[sec]) for sec in cfg.sections() if 'halo' not in sec])
print('baryonic components:', baryons)

R_obs, v_obs = eil.R.to_numpy(float), eil.vc.to_numpy(float)
v_err = 0.5 * (eil.vc_err_minus + eil.vc_err_plus).to_numpy(float)   # symmetrised statistical errors
vbar2 = vcirc(baryons, R_obs)**2                                       # baryonic v_c^2, computed once


def halo(p):
    """NFW halo with log10(M_NFW / Msun) = p[0] and r_s = p[1] kpc."""
    return agama.Potential(type='NFW', mass=10**p[0], scaleRadius=p[1])


def residuals(p, f_bar=1.0):
    """(model - data) / error, for baryons scaled by f_bar plus the NFW halo p."""
    return (np.sqrt(f_bar * vbar2 + vcirc(halo(p), R_obs)**2) - v_obs) / v_err


fit = least_squares(residuals, x0=[12., 15.], bounds=([10, 1], [14, 200]))
chi2, dof = 2 * fit.cost, len(R_obs) - 2           # fit.cost = 0.5 * sum(residuals^2)
pot_fit = agama.Potential(baryons, halo(fit.x))   # a composite potential: baryons + fitted halo
print(f'best fit: log10 M_NFW = {fit.x[0]:.2f}, r_s = {fit.x[1]:.1f} kpc;  chi2 / dof = {chi2:.0f} / {dof} = {chi2 / dof:.1f}')
print(f'v_c(R0) = {vcirc(pot_fit, R0)[0]:.1f} km/s; mass inside 25 kpc = {pot_fit.enclosedMass(25):.3e} Msun')

The fit is good by eye (Figure 11.5, left, orange) but $\chi^2/{\rm dof} \approx 2.7$: the statistical error bars, often below
1 km/s, are smaller than the wiggles of the curve and the systematic errors (2–5%, Eilers et al.). At this precision
the model is limited by our assumptions (axisymmetry, the tracer density, a spherical NFW halo), not by noise.

**The disk–halo degeneracy.** We fixed the baryons. Suppose instead that the disk is lighter or heavier: scale the
baryonic $v_c^2$ by a factor $f_{\rm bar}$ and refit the halo for each value.

In [ ]:
f_grid = np.array([0.4, 0.6, 0.8, 1.0, 1.2])
scan = []
for f in f_grid:
    r = least_squares(residuals, x0=[12., 15.], bounds=([10, 1], [14, 200]), kwargs=dict(f_bar=f))
    M25 = f * baryons.enclosedMass(25) + halo(r.x).enclosedMass(25)   # total mass inside 25 kpc
    scan.append([2 * r.cost, r.x[1], halo(r.x).enclosedMass(25), M25])
    print(f'f_bar = {f:.1f}: chi2 = {2 * r.cost:5.1f}, r_s = {r.x[1]:5.1f} kpc, '
          f'halo mass < 25 kpc = {scan[-1][2]:.2e}, total mass < 25 kpc = {M25:.3e} Msun')
scan = np.array(scan)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].errorbar(R_obs, v_obs, v_err, fmt='ks', ms=3, label='Eilers et al. 2019')
for f in [0.4, 1.0, 1.2]:
    r = least_squares(residuals, x0=[12., 15.], bounds=([10, 1], [14, 200]), kwargs=dict(f_bar=f))
    ax[0].plot(Rgrid, np.sqrt(f * vcirc(baryons, Rgrid)**2 + vcirc(halo(r.x), Rgrid)**2), label=f'total, $f_{{bar}}$ = {f}')
    ax[0].plot(Rgrid, np.sqrt(f) * vcirc(baryons, Rgrid), ':', color=ax[0].lines[-1].get_color())
ax[0].set_xlabel('$R$ [kpc]'); ax[0].set_ylabel('$v_c$ [km/s]'); ax[0].set_ylim(0, 260); ax[0].legend(fontsize=8)
ax[0].set_title('dotted: baryons')
ax[1].plot(f_grid, scan[:, 3] / scan[3, 3], 'ko-', label='total mass < 25 kpc')
ax[1].plot(f_grid, scan[:, 2] / scan[3, 2], 'C0s-', label='halo mass < 25 kpc')
ax[1].plot(f_grid, scan[:, 1] / scan[3, 1], 'C3^-', label='halo scale radius $r_s$')
ax[1].set_yscale('log'); ax[1].set_xlabel('$f_{\\rm bar}$'); ax[1].set_ylabel('relative to $f_{\\rm bar}$ = 1'); ax[1].legend()
plt.tight_layout(); plt.show()

print('chi2 - chi2(f_bar=1):', np.round(scan[:, 0] - scan[3, 0], 1))
print('largest v_bar / v_obs for f_bar = 1.2:', np.max(np.sqrt(1.2 * vbar2) / v_obs).round(3))
assert np.ptp(scan[:, 3]) / scan[3, 3] < 0.1 and scan[:, 1].max() / scan[:, 1].min() > 5

**Figure 11.5.** Left: the Eilers et al. (2019) circular speeds with NFW-halo fits for baryonic mass factors $f_{\rm bar}$ = 0.4, 1.0 and 1.2 (solid: total; dotted: baryons). Right: the total mass and the halo mass inside 25 kpc, and the halo scale radius, as functions of $f_{\rm bar}$, relative to $f_{\rm bar} = 1$.

Between $f_{\rm bar} = 0.4$ and 1 the fits are all acceptable (Figure 11.5, left): $\chi^2$ changes by 14, much less than the excess of
about 60 over the number of degrees of freedom that we already attribute to systematics. The halo adapts: its scale
radius changes by a factor of 3, and by 7 up to $f_{\rm bar} = 1.2$. Heavier baryons are disfavoured ($\Delta\chi^2 = +34$
at 1.2). What stays fixed, within 8%, is the
**total mass inside 25 kpc** (black in Figure 11.5, right), about $2.3\times 10^{11}\,M_\odot$ — the rotation curve measures
$M(<R) \approx R v_c^2/G$, not how it is split between components. Separating disk from halo needs other data:
the vertical motions of stars above the disk (which measure the local disk surface density), microlensing towards the
bulge, or stellar population models. Halo properties far outside 25 kpc (its virial mass, its outer scale) come from
an extrapolation of the NFW form and are not measured by these data at all.

## 11.5 Action space

Finally, the actions of the stars within 200 pc, in two potentials: McMillan (2017) and our fit. Because the
potential is axisymmetric, $J_\phi = L_z$, which is negative in our frame; we plot $-J_\phi$. AGAMA's
`ActionFinder` uses the Stäckel fudge (Chapter 9); `af(xv)` takes an $(N, 6)$ array and returns $(J_r, J_z, J_\phi)$.

In [ ]:
import time
xv_near = np.column_stack([x, y, z, vx, vy, vz])[near]
J = {}
for name, pot in [('McMillan 2017', mcmillan), ('this fit', pot_fit)]:
    af = agama.ActionFinder(pot)
    t0 = time.time()
    J[name] = af(xv_near)                          # columns J_r, J_z, J_phi
    print(f'{name}: actions of {len(xv_near):,} stars in {time.time() - t0:.1f} s')

Jr, Jz, Jphi = J['McMillan 2017'].T
rng_J = [[1400, 2300], [0, 15]]                     # -J_phi [kpc km/s], sqrt(J_r) [sqrt(kpc km/s)]
HJ, _, _ = np.histogram2d(-Jphi, np.sqrt(Jr), bins=200, range=rng_J)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].hist2d(-Jphi, np.sqrt(Jr), bins=300, range=rng_J, norm=LogNorm(), cmap='Greys')
lnHJ = np.log(HJ + 1)
ax[1].imshow((lnHJ - gaussian_filter(lnHJ, 6)).T, origin='lower', extent=[*rng_J[0], *rng_J[1]], aspect='auto',
             cmap='RdBu_r', vmin=-0.8, vmax=0.8)
ax[1].set_title('log counts minus smoothed')
for a in ax:
    a.set_xlabel('$-J_\\phi = -L_z$ [kpc km/s]'); a.set_ylabel('$\\sqrt{J_r}$ [(kpc km/s)$^{1/2}$]')
plt.tight_layout(); plt.show()

ratio_Jr = np.nanmedian(J['this fit'][:, 0] / Jr)    # nan-: unbound stars (a few fast halo stars) get NaN actions
ratio_Jz = np.nanmedian(J['this fit'][:, 1] / Jz)
print('stars with NaN actions:', {k: int(np.isnan(v).any(axis=1).sum()) for k, v in J.items()})
print(f'median ratio of actions, fit / McMillan: J_r {ratio_Jr:.3f}, J_z {ratio_Jz:.3f}')

**Figure 11.6.** Actions of the stars within 200 pc in the McMillan (2017) potential, $(-J_\phi, \sqrt{J_r})$. Left: log-scaled counts. Right: log counts minus a smoothed version (red = excess).

In Figure 11.6 the V shape is the selection: stars within 200 pc of the Sun with $-L_z$ far from that of a circular orbit at $R_0$
($\approx 1850$ kpc km/s) must be on eccentric orbits to reach us, so they need large $J_r$. Inside the V, the moving
groups of §11.2 become **ridges** (Figure 11.6, right): diagonal stripes of excess stars (Trick, Coronado & Rix 2019). A resonance is a
condition on the orbital frequencies, which are functions of the actions, so resonant orbits line up along curves in
action space.

The actions depend on the potential: our fit gives slightly different values from McMillan's model (median ratios
printed above). The structures do not move relative to one another.

**Are these really integrals of motion?** We integrate 200 of the stars for 1 Gyr in the McMillan potential and
recompute their actions. AGAMA's time unit with these units is 1 kpc / (1 km/s) = 0.9778 Gyr.

In [ ]:
sub = xv_near[np.random.default_rng(3).choice(len(xv_near), 200, replace=False)]
orbits = agama.orbit(potential=mcmillan, ic=sub, time=1.0 / 0.9778, trajsize=2, verbose=False)   # trajsize=2: start and end only
xv_end = np.array([traj[-1] for times, traj in orbits])
af = agama.ActionFinder(mcmillan)
J_start, J_end = af(sub), af(xv_end)
rel = np.abs(J_end - J_start) / np.maximum(np.abs(J_start), 1.0)    # relative change (floor 1 kpc km/s)
for k, name in enumerate(['J_r', 'J_z', 'J_phi']):
    print(f'{name:5s}: median relative change {np.median(rel[:, k]):.1e}, 90th percentile {np.percentile(rel[:, k], 90):.1e}')
assert np.all(np.percentile(rel, 90, axis=0) < 0.01)

After 1 Gyr (about four orbits around the Galaxy) the actions changed by less than 0.1% for a typical star and less
than 1% for 90% of them; $J_\phi = L_z$ is conserved exactly in an axisymmetric potential. The Stäckel fudge is an
approximation, but a good one for disk orbits.

## Summary

- Gaia observables become Galactocentric phase-space coordinates with `agama.transformCelestialCoords` and
  `agama.getGalactocentricFromGalactic`; the result depends on the adopted $R_0$, $z_\odot$ and solar velocity.
- The local velocity distribution is clumpy (moving groups), and the $(z, v_z)$ plane shows a phase spiral: the disk
  is not in equilibrium. A shuffle test separates such features from noise.
- The Jeans equation applied to red giants gives the rotation curve; we reproduced Eilers et al. (2019) to 1%.
- A rotation curve measures the enclosed mass, $\approx 2.3\times10^{11}\,M_\odot$ inside 25 kpc, but not how it is shared
  between disk and halo.
- Actions of $10^5$–$10^6$ stars take seconds; moving groups appear as ridges in action space.

| task | AGAMA |
|---|---|
| equatorial → Galactic | `agama.transformCelestialCoords(agama.fromICRStoGalactic, ra, dec, pmra, pmdec)` (radians) |
| Galactic → Galactocentric | `agama.getGalactocentricFromGalactic(l, b, d, pml*4.74, pmb*4.74, vlos, galcen_distance=, z_sun=, galcen_v_sun=)` |
| a model from a file, or part of it | `agama.Potential('McMillan17.ini')`; `agama.Potential(*list_of_dicts)` |
| NFW halo | `agama.Potential(type='NFW', mass=, scaleRadius=)` |
| combine potentials | `agama.Potential(pot1, pot2)` |
| actions | `agama.ActionFinder(pot)(xv)` → $(J_r, J_z, J_\phi)$ |

## Exercises

1. **Solar parameters.** Repeat §11.1–11.2 with the solar velocity of Schönrich, Binney & Dehnen (2010),
   $(U, V, W)_\odot = (11.1, 12.24, 7.25)$ km/s, the Sun's motion relative to a circular orbit; add
   $v_c(R_0)$ from §11.4 to $V_\odot$ to get $v_{y,\odot}$.
   How much do the median $v_R$ and the positions of the moving groups move?
2. **The spiral in $v_R$.** Make the residual map of §11.3 for $v_R$ instead of $v_\phi$. Then split the stars by
   Galactocentric radius ($R < R_0$ and $R > R_0$): does the spiral change shape?
3. **Tracer density.** Redo the rotation curve with a scale length of 2.5 and 3.5 kpc for $\nu$. How large is the
   change in $v_c$ at 8 and at 20 kpc? Compare with the 2% Eilers et al. quote.
4. **Ou et al.** Fit the NFW halo to the Ou et al. (2024) curve instead. How much do the mass inside 25 kpc and the
   implied halo mass inside 100 kpc change? Which of the two is constrained by the data?
5. **Ridges.** Overplot on the action-space map (Figure 11.6) the stars within 30 km/s of the Hercules position of §11.2. Where are
   they?

## References

- T. Antoja et al. (2008), "Origin and evolution of moving groups. I.", A&A 490, 135:
  https://ui.adsabs.harvard.edu/abs/2008A&A...490..135A
- T. Antoja et al. (2018), "A dynamically young and perturbed Milky Way disk", Nature 561, 360:
  https://ui.adsabs.harvard.edu/abs/2018Natur.561..360A
- M. Bennett & J. Bovy (2019), "Vertical waves in the solar neighbourhood in Gaia DR2", MNRAS 482, 1417:
  https://ui.adsabs.harvard.edu/abs/2019MNRAS.482.1417B
- W. Dehnen (1998), "The distribution of nearby stars in velocity space inferred from Hipparcos data", AJ 115, 2384:
  https://ui.adsabs.harvard.edu/abs/1998AJ....115.2384D
- R. Drimmel & E. Poggio (2018), "On the solar velocity", RNAAS 2, 210: https://ui.adsabs.harvard.edu/abs/2018RNAAS...2..210D
- A.-C. Eilers et al. (2019), "The circular velocity curve of the Milky Way from 5 to 25 kpc", ApJ 871, 120:
  https://ui.adsabs.harvard.edu/abs/2019ApJ...871..120E
- Gaia Collaboration, Vallenari et al. (2023), "Gaia Data Release 3", A&A 674, A1:
  https://ui.adsabs.harvard.edu/abs/2023A&A...674A...1G
- GRAVITY Collaboration (2018), A&A 615, L15: https://ui.adsabs.harvard.edu/abs/2018A&A...615L..15G
- D. W. Hogg, A.-C. Eilers & H.-W. Rix (2019), "Spectrophotometric parallaxes with linear models", AJ 158, 147:
  https://ui.adsabs.harvard.edu/abs/2019AJ....158..147H
- P. McMillan (2017), "The mass distribution and gravitational potential of the Milky Way", MNRAS 465, 76:
  https://ui.adsabs.harvard.edu/abs/2017MNRAS.465...76M
- X. Ou et al. (2024), "The dark matter profile of the Milky Way inferred from its circular velocity curve",
  MNRAS 528, 693: https://ui.adsabs.harvard.edu/abs/2024MNRAS.528..693O
- R. Schönrich, J. Binney & W. Dehnen (2010), "Local kinematics and the local standard of rest", MNRAS 403, 1829:
  https://ui.adsabs.harvard.edu/abs/2010MNRAS.403.1829S
- W. Trick, J. Coronado & H.-W. Rix (2019), "The Galactic disc in action space as seen by Gaia DR2", MNRAS 484, 3291:
  https://ui.adsabs.harvard.edu/abs/2019MNRAS.484.3291T
- Data: V. Belokurov (2026), "Galactic Dynamics Tutorial data", Zenodo, https://doi.org/10.5281/zenodo.23627795 — please also cite the sources listed in its README
  (Gaia DR3, Andrae et al. 2023, Katz et al. 2023, APOGEE DR17).